##### Setup

###### 1. Importações

In [2]:
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display

###### 2. Pontos homogêneos

In [3]:
# Usada nas cinco questões: pontos Nx2 ou Nx3 viram colunas homogêneas.
def pontos_homogeneos(coordenadas):
    coordenadas = np.asarray(coordenadas, dtype=float)
    if coordenadas.ndim != 2 or coordenadas.shape[1] not in (2, 3):
        raise ValueError("Use uma matriz Nx2 ou Nx3 de coordenadas")
    return np.vstack((coordenadas.T, np.ones(coordenadas.shape[0])))

###### 3. Translação 2D

In [4]:
# Compartilhada pelas questões 2 e 3.
def translacao_2d(tx, ty):
    return np.array([[1, 0, tx],
                     [0, 1, ty],
                     [0, 0, 1]], dtype=float)

###### 4. Gráficos cartesianos 2D

In [5]:
# O plano é usado nas questões 1, 2, 3 e 5.
def configurar_plano(eixo, titulo):
    eixo.axhline(0, color="#777777", linewidth=1)
    eixo.axvline(0, color="#777777", linewidth=1)
    eixo.grid(True, color="#dddddd", linewidth=0.8)
    eixo.set_axisbelow(True)
    eixo.set_aspect("equal", adjustable="box")
    eixo.set(xlabel="x", ylabel="y", title=titulo)
    eixo.margins(0.15)


# A comparação é usada nas questões 1, 2 e 3.
def comparar_pontos_2d(originais, transformados, titulo, rotulo,
                       arquivo, contorno=None, centro=None):
    figura, eixo = plt.subplots(figsize=(6, 6))
    if contorno is not None:
        eixo.plot(originais[0, contorno], originais[1, contorno],
                  color="tab:blue", alpha=0.7)
        eixo.plot(transformados[0, contorno], transformados[1, contorno],
                  color="tab:red", alpha=0.7)
    eixo.scatter(originais[0], originais[1], color="tab:blue",
                 s=60, label="Original", zorder=3)
    eixo.scatter(transformados[0], transformados[1], color="tab:red",
                 s=60, label=rotulo, zorder=3)
    if centro is not None:
        eixo.scatter(*centro, marker="x", color="black", s=100,
                     label="Centro de rotação", zorder=4)
    configurar_plano(eixo, titulo)
    eixo.legend()
    figura.tight_layout()
    figura.savefig(arquivo, dpi=160, bbox_inches="tight")
    plt.show()
    plt.close(figura)


# Usada em todas as questões: os valores iniciais executam o exemplo do PDF.
def exibir_controles(funcao, controles, extras=()):
    saida = widgets.interactive_output(funcao, controles)
    display(widgets.VBox([*extras, *controles.values()]), saida)


##### QUESTÃO1

###### 1. 

In [6]:
pontos = pontos_homogeneos([[2, 1], [3, 1], [2, 2], [3, 2]])


def mostrar_escala(sx, sy):
    escala = np.diag([sx, sy, 1.0])
    print("Matriz de escala:\n", escala)
    comparar_pontos_2d(
        pontos, escala @ pontos, "Questão 1 — Escala 2D", "Escalado",
        "lista3_questao1_escala.png", contorno=[0, 1, 3, 2, 0],
    )


exibir_controles(mostrar_escala, {
    "sx": widgets.FloatSlider(value=3, min=-4, max=6, step=0.25,
                              description="sx", continuous_update=False),
    "sy": widgets.FloatSlider(value=4, min=-4, max=6, step=0.25,
                              description="sy", continuous_update=False),
})

Output()

##### QUESTÃO2

###### 1. 

In [7]:
ponto_inicial = pontos_homogeneos([[3, 2]])


def mostrar_translacao(tx, ty):
    ponto_final = translacao_2d(tx, ty) @ ponto_inicial
    print("Ponto original:", ponto_inicial[:2, 0])
    print("Ponto transladado:", ponto_final[:2, 0])
    comparar_pontos_2d(
        ponto_inicial, ponto_final, "Questão 2 — Translação 2D", "Transladado",
        "lista3_questao2_translacao.png",
    )


exibir_controles(mostrar_translacao, {
    "tx": widgets.FloatSlider(value=3, min=-10, max=10, step=0.5,
                              description="tx", continuous_update=False),
    "ty": widgets.FloatSlider(value=6, min=-10, max=10, step=0.5,
                              description="ty", continuous_update=False),
})

Output()

##### QUESTÃO3

###### 1. 

In [8]:
def rotacao_2d(angulo):
    c, s = np.cos(angulo), np.sin(angulo)
    return np.array([[c, -s, 0],
                     [s,  c, 0],
                     [0,  0, 1]], dtype=float)


quadrado = pontos_homogeneos([[1, 1], [2, 1], [2, 2], [1, 2]])


def mostrar_rotacao_origem(graus):
    transformados = rotacao_2d(np.deg2rad(graus)) @ quadrado
    comparar_pontos_2d(
        quadrado, transformados,
        f"Questão 3a — Rotação de {graus}° na origem", "Rotacionado",
        "lista3_questao3a_origem.png", contorno=[0, 1, 2, 3, 0],
    )


exibir_controles(mostrar_rotacao_origem, {
    "graus": widgets.IntSlider(value=45, min=-180, max=180, step=5,
                               description="Ângulo", continuous_update=False),
})

Output()

###### 2. 

In [9]:
def mostrar_rotacao_ponto(graus, cx, cy):
    angulo = np.deg2rad(graus)
    rotacao_no_centro = (translacao_2d(cx, cy) @ rotacao_2d(angulo)
                         @ translacao_2d(-cx, -cy))
    comparar_pontos_2d(
        quadrado, rotacao_no_centro @ quadrado,
        f"Questão 3b — Rotação de {graus}° em torno de ({cx:g}, {cy:g})",
        "Rotacionado", "lista3_questao3b_centro.png",
        contorno=[0, 1, 2, 3, 0], centro=(cx, cy),
    )


exibir_controles(mostrar_rotacao_ponto, {
    "graus": widgets.IntSlider(value=45, min=-180, max=180, step=5,
                               description="Ângulo", continuous_update=False),
    "cx": widgets.FloatSlider(value=1, min=-3, max=3, step=0.25,
                              description="cx", continuous_update=False),
    "cy": widgets.FloatSlider(value=2, min=-3, max=3, step=0.25,
                              description="cy", continuous_update=False),
})

Output()

##### QUESTÃO4

###### 1. 

Usamos a malha [Suzanne em OBJ](https://github.com/alecjacobson/common-3d-test-models/blob/master/data/suzanne.obj), incluída em `Archives/suzanne.obj`. No Colab, a célula baixa o arquivo se ele ainda não estiver disponível. O olho esquerdo é uma parte separada da malha; as orelhas são deslocadas de forma gradual a partir das bases. Todas as transformações e a animação são feitas em NumPy e Matplotlib. O botão Reproduzir move orelhas e olho juntos; os controles também permitem ajustar o olhar e a câmera.

In [10]:
from pathlib import Path
from urllib.request import urlopen
from mpl_toolkits.mplot3d.art3d import Poly3DCollection


# O arquivo faz parte do repositório; o download atende notebooks abertos
# isoladamente no Colab, onde as pastas do GitHub não são copiadas.
caminho_modelo = Path("Archives/suzanne.obj")
if not caminho_modelo.is_file():
    fonte = ("https://raw.githubusercontent.com/alecjacobson/"
             "common-3d-test-models/master/data/suzanne.obj")
    caminho_modelo = Path("suzanne.obj")
    try:
        with urlopen(fonte, timeout=30) as resposta:
            caminho_modelo.write_bytes(resposta.read())
    except OSError as erro:
        raise RuntimeError("Não foi possível baixar o modelo Suzanne") from erro


# OBJ contém vértices (v) e faces (f); normais e materiais não são necessários.
vertices_obj, faces = [], []
with caminho_modelo.open(encoding="utf-8") as arquivo_obj:
    for linha in arquivo_obj:
        if linha.startswith("v "):
            vertices_obj.append([float(valor) for valor in linha.split()[1:4]])
        elif linha.startswith("f "):
            faces.append(np.array([
                int(parte.split("/")[0]) - 1 for parte in linha.split()[1:]
            ], dtype=int))

vertices_obj = np.asarray(vertices_obj, dtype=float)
if len(vertices_obj) == 0 or len(faces) == 0:
    raise ValueError("O arquivo OBJ não contém vértices e faces")

# Centraliza Suzanne e coloca a frente do rosto na direção de Y negativo.
centro_modelo = (vertices_obj.min(axis=0) + vertices_obj.max(axis=0)) / 2
vertices = (vertices_obj - centro_modelo)[:, [0, 2, 1]]
vertices[:, 1] *= -1


# As duas malhas pequenas desconectadas do rosto são os olhos.
adjacentes = [set() for _ in vertices]
for face in faces:
    for atual, proximo in zip(face, np.roll(face, -1)):
        adjacentes[atual].add(proximo)
        adjacentes[proximo].add(atual)

componentes = []
visitados = set()
for vertice in range(len(vertices)):
    if vertice in visitados:
        continue
    pilha = [vertice]
    componente = []
    visitados.add(vertice)
    while pilha:
        atual = pilha.pop()
        componente.append(atual)
        for vizinho in adjacentes[atual] - visitados:
            visitados.add(vizinho)
            pilha.append(vizinho)
    componentes.append(np.array(componente, dtype=int))

olhos = sorted((parte for parte in componentes if 20 <= len(parte) <= 50),
              key=lambda parte: vertices[parte, 0].mean())
if len(olhos) != 2:
    raise ValueError("Não foi possível identificar os dois olhos de Suzanne")
olho_esquerdo, olho_direito = olhos
centro_olho = vertices[olho_esquerdo].mean(axis=0)
centro_olho_direito = vertices[olho_direito].mean(axis=0)
pupila_esquerda = centro_olho + [0, -0.16, 0]
pupila_direita = centro_olho_direito + [0, -0.16, 0]

indices_olhos = set(np.concatenate(olhos))
cores_faces = ["#f7f3e9" if face[0] in indices_olhos else "#c79c6e"
               for face in faces]
pontos_4d = pontos_homogeneos(vertices)  # Função compartilhada do Setup.


# Rotação e translação 3D são usadas apenas nesta questão.
def translacao_3d(tx, ty, tz):
    matriz = np.eye(4)
    matriz[:3, 3] = (tx, ty, tz)
    return matriz


def rotacao_y(angulo):
    c, s = np.cos(angulo), np.sin(angulo)
    return np.array([[c, 0, s, 0], [0, 1, 0, 0],
                     [-s, 0, c, 0], [0, 0, 0, 1]], dtype=float)


def rotacao_x(angulo):
    c, s = np.cos(angulo), np.sin(angulo)
    return np.array([[1, 0, 0, 0], [0, c, -s, 0],
                     [0, s, c, 0], [0, 0, 0, 1]], dtype=float)


def girar_em_torno(pontos, centro, rotacao):
    x, y, z = centro
    matriz = (translacao_3d(x, y, z) @ rotacao
              @ translacao_3d(-x, -y, -z))
    return matriz @ pontos


def pose_suzanne(angulo_orelhas, angulo_olho):
    pose = vertices.copy()

    # Uma faixa suave entre cabeça e orelha evita abrir a malha na junção.
    distancia = np.abs(vertices[:, 0])
    intensidade = np.clip((distancia - 0.78) / 0.34, 0, 1)
    intensidade = intensidade * intensidade * (3 - 2 * intensidade)
    for lado in (-1, 1):
        mascara = vertices[:, 0] * lado > 0.78
        base = (lado * 0.78, 0, 0)
        girados = girar_em_torno(pontos_4d[:, mascara], base,
                                 rotacao_y(lado * angulo_orelhas))[:3].T
        peso = intensidade[mascara, None]
        pose[mascara] = (1 - peso) * vertices[mascara] + peso * girados

    # O olho esquerdo e sua pupila giram; o direito permanece fixo.
    rotacao_olho = rotacao_x(angulo_olho)
    pose[olho_esquerdo] = girar_em_torno(
        pontos_4d[:, olho_esquerdo], centro_olho, rotacao_olho
    )[:3].T
    pupila = girar_em_torno(
        pontos_homogeneos([pupila_esquerda]), centro_olho, rotacao_olho
    )[:3, 0]
    return pose, pupila


def desenhar_pose(eixo, angulo_orelhas, angulo_olho, elevacao, azimute):
    eixo.clear()
    pose, pupila = pose_suzanne(angulo_orelhas, angulo_olho)
    malha = Poly3DCollection([pose[face] for face in faces],
                             facecolors=cores_faces, edgecolor="#654c3f",
                             linewidth=0.22, zorder=1)
    eixo.add_collection3d(malha)
    eixo.scatter(*pupila, color="black", s=38, depthshade=False, zorder=3)
    eixo.scatter(*pupila_direita, color="black", s=38, depthshade=False, zorder=3)
    eixo.set(xlim=(-1.5, 1.5), ylim=(-1, 1), zlim=(-1.1, 1.1),
             title="Questão 4 — Suzanne: orelhas e olho")
    eixo.set_box_aspect((3, 2, 2.2))
    eixo.view_init(elev=elevacao, azim=azimute)
    eixo.set_axis_off()


def mostrar_suzanne(fase, olhar, elevacao, azimute):
    figura = plt.figure(figsize=(7, 6))
    eixo_3d = figura.add_subplot(111, projection="3d", computed_zorder=False)
    angulo_orelhas = fase / 20
    desenhar_pose(eixo_3d, angulo_orelhas, angulo_orelhas + olhar,
                 elevacao, azimute)
    figura.savefig("lista3_questao4_orelha_olho.png", dpi=150,
                  bbox_inches="tight")
    plt.show()
    plt.close(figura)


controles_suzanne = {
    "fase": widgets.IntSlider(value=0, min=-10, max=10, step=1,
                              description="Orelhas", continuous_update=False),
    "olhar": widgets.FloatSlider(value=0, min=-0.5, max=0.5, step=0.05,
                                 description="Olhar", continuous_update=False),
    "elevacao": widgets.IntSlider(value=10, min=-30, max=70, step=5,
                                  description="Elevação", continuous_update=False),
    "azimute": widgets.IntSlider(value=-90, min=-180, max=180, step=5,
                                 description="Azimute", continuous_update=False),
}
reproduzir = widgets.Play(value=0, min=-10, max=10, step=1, interval=450,
                         description="Reproduzir")
ligacao_reproducao = widgets.link((reproduzir, "value"),
                                 (controles_suzanne["fase"], "value"))
exibir_controles(mostrar_suzanne, controles_suzanne, extras=(reproduzir,))


Output()

##### QUESTÃO5

###### 1. 

Para a câmera em `(0, 0, -D)` e o plano de imagem em `z = 0`, usamos `H = 1 + Z/D`. O exemplo impresso usa `H = Z/D`; com o cubo centrado na origem, isso faz as faces opostas coincidirem na projeção.

In [11]:
def mostrar_perspectiva(distancia_camera, tamanho_cubo):
    t = tamanho_cubo
    vertices_cubo = pontos_homogeneos([
        [-t, -t, -t], [t, -t, -t], [t, t, -t], [-t, t, -t],
        [-t, -t,  t], [t, -t,  t], [t, t,  t], [-t, t,  t],
    ])
    perspectiva = np.array([
        [1, 0, 0, 0], [0, 1, 0, 0], [0, 0, 1, 0],
        [0, 0, 1 / distancia_camera, 1],
    ], dtype=float)
    projetados = perspectiva @ vertices_cubo
    xy = projetados[:2] / projetados[3]
    arestas = [(0, 1), (1, 2), (2, 3), (3, 0),
               (4, 5), (5, 6), (6, 7), (7, 4),
               (0, 4), (1, 5), (2, 6), (3, 7)]

    figura, eixo = plt.subplots(figsize=(6, 6))
    for inicio, fim in arestas:
        eixo.plot(xy[0, [inicio, fim]], xy[1, [inicio, fim]],
                  color="tab:blue", linewidth=2)
    eixo.scatter(xy[0], xy[1], color="tab:red", zorder=3)
    for indice, (x, y) in enumerate(xy.T):
        eixo.annotate(str(indice), (x, y), xytext=(5, 5),
                      textcoords="offset points")
    configurar_plano(eixo, "Questão 5 — Cubo em perspectiva")
    figura.tight_layout()
    figura.savefig("lista3_questao5_perspectiva.png", dpi=160,
                  bbox_inches="tight")
    plt.show()
    plt.close(figura)


exibir_controles(mostrar_perspectiva, {
    "distancia_camera": widgets.FloatSlider(
        value=5, min=2.5, max=12, step=0.5,
        description="D", continuous_update=False),
    "tamanho_cubo": widgets.FloatSlider(
        value=1, min=0.5, max=2, step=0.1,
        description="Tamanho", continuous_update=False),
})

Output()